In [1]:
import pandas as pd
model_df = pd.read_csv('amr_modeling_dataset.csv')
mut_cols = [c for c in model_df.columns if c.endswith('_mut')]
print(model_df.shape)
model_df.head()

(986, 23)


,accession,allele,label,seq_len,coverage,pos_245,pos_553,pos_562,pos_593,pos_600,...,pos_245_mut,pos_553_mut,pos_562_mut,pos_593_mut,pos_600_mut,pos_601_mut,pos_676_mut,pident,length,any_hotspot_mut
0,GCA_057694315.1,1,safe,731,731,T,T,S,M,G,...,0,0,0,0,0,0,0,100.0,349,False
1,GCA_057779585.1,1,safe,725,725,T,T,S,M,G,...,0,0,0,0,0,0,0,100.0,349,False
2,GCA_057779485.1,1,safe,707,707,T,T,S,M,G,...,0,0,0,0,0,0,0,100.0,349,False
3,GCA_057694075.1,1,safe,751,751,T,T,S,M,G,...,0,0,0,0,0,0,0,100.0,349,False
4,GCA_057779105.1,1,safe,704,704,T,T,S,M,G,...,0,0,0,0,0,0,0,100.0,349,False


In [5]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# train ONLY on consensus wild-type isolates (the "normal" baseline)
safe_mask = (y == 0)
iso_forest = IsolationForest(contamination=0.1, random_state=42)
iso_forest.fit(X[safe_mask])

# score everyone (including mutants the model never trained on)
anomaly_scores = -iso_forest.score_samples(X)  # higher = more anomalous
iso_pred = (iso_forest.predict(X) == -1).astype(int)  # -1 = anomaly = flag

print("Precision:", round(precision_score(y, iso_pred), 3))
print("Recall:", round(recall_score(y, iso_pred), 3))
print("F1:", round(f1_score(y, iso_pred), 3))
print("AUROC:", round(roc_auc_score(y, anomaly_scores), 3))

Precision: 0.843
Recall: 0.989
F1: 0.91
AUROC: 0.925


In [6]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_preds = np.zeros(len(y))
all_scores = np.zeros(len(y))

for train_idx, test_idx in skf.split(X, y):
    # fit ONLY on safe isolates within this fold's training portion
    train_safe_idx = train_idx[y.iloc[train_idx].values == 0]
    iso = IsolationForest(contamination=0.1, random_state=42)
    iso.fit(X.iloc[train_safe_idx])

    all_preds[test_idx] = (iso.predict(X.iloc[test_idx]) == -1).astype(int)
    all_scores[test_idx] = -iso.score_samples(X.iloc[test_idx])

print("Precision:", round(precision_score(y, all_preds), 3))
print("Recall:", round(recall_score(y, all_preds), 3))
print("F1:", round(f1_score(y, all_preds), 3))
print("AUROC:", round(roc_auc_score(y, all_scores), 3))

Precision: 0.843
Recall: 0.989
F1: 0.91
AUROC: 0.928


In [7]:
pos_cols_raw = [f'pos_{p}' for p in [245, 553, 562, 593, 600, 601, 676]]

# check if the raw (non-binary) position columns are still in your CSV
print([c for c in model_df.columns if c in pos_cols_raw])

['pos_245', 'pos_553', 'pos_562', 'pos_593', 'pos_600', 'pos_601', 'pos_676']


In [8]:
CONSENSUS_WT = {}
for col in pos_cols_raw:
    p = int(col.split('_')[1])
    CONSENSUS_WT[p] = model_df[col].mode(dropna=True).iloc[0]
print(CONSENSUS_WT)

{245: 'T', 553: 'T', 562: 'S', 593: 'M', 600: 'G', 601: 'P', 676: 'P'}


In [9]:
HOTSPOTS = [245, 553, 562, 593, 600, 601, 676]

In [12]:
from Bio import SeqIO

seqs = {r.id: str(r.seq) for r in SeqIO.parse('isolate_pbp2x.faa', 'fasta')}
lengths = {acc: len(s) for acc, s in seqs.items()}

ref_acc = max(lengths, key=lengths.get)
ref_seq = seqs[ref_acc]
print("Reference:", ref_acc, "| length:", len(ref_seq))

Reference: GCA_057694075.1 | length: 751


In [13]:
final_iso_model = IsolationForest(contamination=0.1, random_state=42)
final_iso_model.fit(X[safe_mask])

artifact = {
    'model': final_iso_model,
    'model_type': 'isolation_forest',
    'consensus_wt': CONSENSUS_WT,
    'hotspots': HOTSPOTS,
    'feature_cols': mut_cols,
    'ref_seq': ref_seq,
    'cv_metrics': {'precision': 0.843, 'recall': 0.989, 'f1': 0.91, 'auroc': 0.928},
    'rf_comparison_note': 'Random Forest achieved AUROC 0.99 in-distribution but 0.01-0.02 under leave-one-group-out; Isolation Forest selected for deployment.'
}
import pickle
with open('pbp2x_flag_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)
from google.colab import files
files.download('pbp2x_flag_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>